# Competition notebook: Qwen two-stage LoRA

# Qwen-only reproduction of the proven two-stage LoRA pipeline

This version keeps the public-pool n-gram model, Qwen3-8B and Qwen3-14B two-stage LoRA checkpoints, centered twin, label calibration, sibling specialists, and balancing. OLMo2 and SmolLM3 components are intentionally excluded for an isolated first run.

# Two-stage LoRA on the public pool (LB 0.592)

**The problem in one line.** Telling Claude from GPT is easy. Telling `gemini 3.7 flash` from `gemini 3.8 flash` is the whole game, and the host gives us only 150 passages per model to learn it from.

**The idea.** Other Kagglers have published about 35,000 passages from these same 18 models, written to prompts in the competition's style. That is 13 times the training set. So we teach in two stages, the way you'd learn a language:

1. **Read a lot first.** Train a LoRA classifier on the public passages only. It learns what each model *sounds like*.
2. **Then listen to the teacher.** Fine tune it for 2 epochs on the host's 2,699 passages. It learns how each model sounded *when the host asked*: same settings, same formatting.

Stage 1 alone already matches a LoRA trained on the host data. Stage 2 adds another 9 points.

| Model | Local CV (5-fold, grouped by prompt) | Public LB |
|---|---|---|
| n-gram SVC on train + pool | 0.435 | |
| Qwen3-8B LoRA, public passages only (stage 1) | 0.483 | |
| Qwen3-8B LoRA, stage 1 + stage 2 | 0.569 | |
| Qwen3-14B LoRA, stage 1 + stage 2 | 0.569 | |
| **This notebook:** 8B + 14B + n-gram, centred twin, balancing | **0.593** | **0.592** |

The two LoRAs score the same alone but disagree on 22% of passages, which is why blending them helps.

**What runs here** (inference only, about 1 h on 2x T4 for the full test):
1. An n-gram model, fit here on train + pool.
2. Two LoRA checkpoints (Qwen3-8B and Qwen3-14B), trained offline. The training recipe is at the end.
3. A "centred twin" that compares passages answering the same question.
4. A log-probability blend, then class balancing.


In [1]:
import gc, glob, json, os, re, time
import numpy as np
import pandas as pd
import torch
from torch import nn

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights, fit on out-of-fold predictions (log probabilities are multiplied by these and summed).
W_SVC = 1.021
LORAS = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model to look for
    "lora8b_v1":  dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)),
    "lora14b_v1": dict(w=0.429, twin_w=0.150, bs=8,  base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
}
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # retained for diagnostics; contribution disabled below

# Column order of the checkpoints' heads (checked against each checkpoint's meta.json below).
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))   # every member adds w * log p here
print("train", train.shape, "test", test.shape, flush=True)


train (2699, 4) test (3, 2)


## 1. The n gram model

Each model has typing habits. In the training data claude and GLM never use curly quotes, while gpt uses a curly apostrophe in about half its passages. Gemini almost never uses an em dash, while Claude uses one in three passages out of four. Character n grams pick this up.

We look at each passage four ways: raw characters +  words plus punctuation +  the *shape* of the text (`Xxxx xx x`) + function words only, with every content word replaced by `CONTENT` so the topic can't leak in. A linear SVM sits on top, trained on train + the public pool with balanced class weights.


In [2]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]
def shape_text(t):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)
def function_text(t):
    return " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))
pool = pd.read_parquet(find("pool_v1.parquet"))

# Optional external augmentation.  The notebook remains runnable without it.
# Keep only in-style, competition-valid passages and give them a conservative weight.
extra_hits = sorted(glob.glob(f"{INPUT_DIR}/**/passages_clean.csv", recursive=True))
extra = pd.DataFrame(columns=["text", "model"])
if extra_hits:
    extra = pd.read_csv(extra_hits[0], usecols=lambda c: c in {"text", "model"})
    extra = extra.dropna(subset=["text", "model"]).copy()
    extra["model"] = extra["model"].astype(str)
    extra = extra[extra["model"].isin(LABELS)].copy()
    extra = extra[~extra["model"].isin({"kimi-k3", "gpt-5.6-luna"})].copy()
    extra["text"] = extra["text"].astype(str)
    word_count = extra["text"].str.split().str.len()
    extra = extra[word_count.between(40, 250)].copy()
    known_text = set(train["text"].astype(str)) | set(pool["text"].astype(str))
    extra = extra[~extra["text"].isin(known_text)].drop_duplicates("text")

extra_texts = extra["text"].tolist()
extra_y = extra["model"].map({l: i for i, l in enumerate(LABELS)}).astype(int).values
fit_texts = train.text.astype(str).tolist() + pool.text.astype(str).tolist() + extra_texts
fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values, extra_y]
fit_w = np.r_[np.ones(len(train), dtype=np.float32),
              np.full(len(pool), 0.50, dtype=np.float32),
              np.full(len(extra), 0.35, dtype=np.float32)]
print(f"SVC rows: train={len(train):,}, pool={len(pool):,}, clean_extra={len(extra):,}", flush=True)
A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else function_text if view == "function" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "function" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test.text]) * weight)
svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(
    hstack(A, format="csr"), fit_y, sample_weight=fit_w)
P_svc = softmax(svc.decision_function(hstack(B, format="csr")))
P_SVC_TEST = P_svc  # blended after guarded grouped-OOF weight selection
del A, B, svc, pool, extra
gc.collect()
print(f"n-gram model done ({time.time() - T0:.0f}s)", flush=True)


n-gram model done (205s)


## 2. The LoRA classifiers

Each checkpoint is a LoRA adapter on the **first 24 layers of Qwen3 8B** (or the first 27 of Qwen3 14B). We don't need the rest: style lives in the early and middle layers, while the last layers mostly encode the topic, which is useless here because test prompts are new.

For each passage we:
1. merge the adapter into the weights once (`W <- W + B A * alpha / r`), so no `peft` is needed
2. run the passage and take the mean hidden state at 8 depths (layers 2, 4, 6, 8, 12, 16, 20, 24), each through its own LayerNorm;
3. concatenate them (8 x 4096 numbers) and apply a linear layer to get 18 scores.

The final RMSNorm is switched off (`Identity`) because the checkpoints were trained that way. On a T4 everything runs in fp16, a batch that overflows is redone in bf16.


In [3]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """First `keep` decoder layers, split over both T4s when it does not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    hd = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)
    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd
               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." in k:
            name = k.split(".lora_A.")[0]
            lin = modules[name[name.index("layers."):]]
            Bm = sd[k.replace(".lora_A.", ".lora_B.")]
            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z (n, 8 * hidden) and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    ids = [x + [tok.eos_token_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together = less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()
    del model
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}


## 3. Compare answers to the same question

Picture two test passages that answer the same prompt. Whatever they share, the topic, is noise for us. Subtract the pair's average and what's left is mostly *how* each one was written.

So we cluster the test passages by topic (content word TFIDF, average linkage). In each cluster of 2 to 18 passages, we subtract the cluster mean from every passage's LoRA features and read them with a second head, which was trained during fine-tuning on exactly such differences. It adds a little evidence on top of the plain head.

How much this helps depends on how many test passages share a prompt, which we don't know. In simulation it gives +0.5 points at 1-2 passages per prompt and more with larger groups. With no clusters it simply does nothing.


In [4]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
np.fill_diagonal(D, 0)
groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance") \
    if len(test) >= 2 else np.ones(len(test), dtype=int)
size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
del D
print(f"topic clusters: {int((size == 1).sum())} singletons, {int(twin_rows.sum())} passages in clusters of 2-18", flush=True)

# Guarded grouped-OOF stacking. The search is deliberately conservative:
# if it does not beat the reference blend on prompt-grouped OOF, retain the
# original weights and avoid fitting the test set to a noisy meta-model.
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
def _oof_svc_probs():
    oof = np.zeros((len(train), K), np.float32)
    groups_oof = train["prompt_id"].astype(str) if "prompt_id" in train else np.arange(len(train))
    for fold, (itr, iva) in enumerate(GroupKFold(3).split(train, y, groups_oof), 1):
        AA, BB = [], []
        tr_text = train.text.iloc[itr].astype(str).tolist()
        va_text = train.text.iloc[iva].astype(str).tolist()
        for view, lo, hi, max_features, weight in VIEWS:
            prep = shape_text if view == "shape" else function_text if view == "function" else (lambda s: s)
            analyzer = "char" if view == "shape" else "word" if view == "function" else view
            opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                        max_features=max_features, dtype=np.float32)
            if analyzer == "word": opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
            vec = TfidfVectorizer(**opts)
            AA.append(vec.fit_transform([prep(t) for t in tr_text]) * weight)
            BB.append(vec.transform([prep(t) for t in va_text]) * weight)
        fold_model = LinearSVC(C=0.5, random_state=100 + fold, max_iter=4000, class_weight="balanced")
        fold_model.fit(hstack(AA, format="csr"), y[itr])
        oof[iva] = softmax(fold_model.decision_function(hstack(BB, format="csr")))
        print(f"OOF SVC fold {fold}/3 done", flush=True)
    return oof
svc_oof = _oof_svc_probs()
all_text = train.text.astype(str).tolist() + test.text.astype(str).tolist()
q_train, q_test = {}, {}
for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z_all, h_all = lora_features(ckpt, find_base(mc["base"]), all_text, mc["bs"])
    q_all = softmax(Z_all @ h_all["plain.weight"].T + h_all["plain.bias"])
    q_train[ckpt_name], q_test[ckpt_name] = q_all[:len(train)], q_all[len(train):]
    del Z_all, h_all
    gc.collect()
ref = (W_SVC, LORAS["lora8b_v1"]["w"], LORAS["lora14b_v1"]["w"])
ref_ba = balanced_accuracy_score(y, (ref[0] * np.log(np.clip(svc_oof, 1e-9, 1))
    + ref[1] * np.log(np.clip(q_train["lora8b_v1"], 1e-9, 1))
    + ref[2] * np.log(np.clip(q_train["lora14b_v1"], 1e-9, 1))).argmax(1))
best = (ref_ba, *ref)
for ws in (0.75, 1.0, 1.25):
    for w8 in (0.20, 0.30, 0.40):
        for w14 in (0.35, 0.45, 0.55):
            pred = ws * np.log(np.clip(svc_oof, 1e-9, 1)) + w8 * np.log(np.clip(q_train["lora8b_v1"], 1e-9, 1)) + w14 * np.log(np.clip(q_train["lora14b_v1"], 1e-9, 1))
            ba = balanced_accuracy_score(y, pred.argmax(1))
            if ba > best[0]: best = (ba, ws, w8, w14)
if best[0] > ref_ba + 0.002:
    _, W_SVC, LORAS["lora8b_v1"]["w"], LORAS["lora14b_v1"]["w"] = best
print(f"OOF blend reference={ref_ba:.5f}, selected={best}", flush=True)
LOGIT += W_SVC * np.log(np.clip(P_SVC_TEST, 1e-9, 1))

# Optional Qwen3 embedding-SVM phase. The attachment is not required: when
# the embedding model is absent, the proven OOF blend runs unchanged.
embed_configs = sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True))
embed_hits = [os.path.dirname(x) for x in embed_configs
              if "qwen" in x.lower() and "embedding" in x.lower()]
if embed_hits:
    from sklearn.model_selection import GroupKFold
    import torch.nn.functional as F
    embed_dir = embed_hits[0]
    print(f"embedding model found: {embed_dir}", flush=True)
    embed_tok = AutoTokenizer.from_pretrained(embed_dir, padding_side="left", local_files_only=True)
    embed_model = AutoModel.from_pretrained(embed_dir, torch_dtype=torch.float16,
                                             local_files_only=True).cuda().eval()
    def _embed(items, bs=8, max_len=512):
        out = []
        for start in range(0, len(items), bs):
            tok = embed_tok(items[start:start + bs], padding=True, truncation=True,
                            max_length=max_len, return_tensors="pt")
            tok = {k: v.cuda() for k, v in tok.items()}
            with torch.inference_mode():
                h = embed_model(**tok).last_hidden_state
                mask = tok["attention_mask"]
                left = bool((mask[:, -1].sum() == mask.shape[0]).item())
                if left:
                    pooled = h[:, -1]
                else:
                    pos = mask.sum(1) - 1
                    pooled = h[torch.arange(h.shape[0], device=h.device), pos]
                out.append(F.normalize(pooled.float(), p=2, dim=1).cpu().numpy())
        return np.vstack(out).astype(np.float32)
    emb_all = _embed(train.text.astype(str).tolist() + test.text.astype(str).tolist())
    emb_train, emb_test = emb_all[:len(train)], emb_all[len(train):]
    emb_groups = train["prompt_id"].astype(str) if "prompt_id" in train else np.arange(len(train))
    emb_scores = {}
    for C_emb in (0.3, 1.0, 3.0):
        fold_scores = []
        for itr, iva in GroupKFold(3).split(emb_train, y, emb_groups):
            em = LinearSVC(C=C_emb, class_weight="balanced", max_iter=3000, random_state=31)
            em.fit(emb_train[itr], y[itr])
            fold_scores.append(balanced_accuracy_score(y[iva], em.predict(emb_train[iva])))
        emb_scores[C_emb] = float(np.mean(fold_scores))
    best_C_emb = max(emb_scores, key=emb_scores.get)
    emb_model = LinearSVC(C=best_C_emb, class_weight="balanced", max_iter=5000, random_state=31)
    emb_model.fit(emb_train, y)
    P_EMB = softmax(emb_model.decision_function(emb_test))
    LOGIT += 0.18 * np.log(np.clip(P_EMB, 1e-9, 1))
    print(f"embedding-SVM grouped BA={emb_scores}, selected_C={best_C_emb}", flush=True)
    del embed_model, embed_tok, emb_all, emb_train, emb_test, emb_model
    torch.cuda.empty_cache(); gc.collect()
else:
    print("embedding model not attached; embedding-SVM phase skipped", flush=True)

for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])
    LOGIT += mc["w"] * np.log(np.clip(softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), 1e-9, 1))
    if twin_rows.any():
        # Leave-one-out cavity centering: do not let each passage cancel
        # itself out of the topic mean. Shrink the larger n=2 correction so
        # its scale remains comparable to the original centred head.
        Zc = np.zeros_like(Z)
        for g in np.unique(groups[twin_rows]):
            idx = np.flatnonzero(groups == g)
            group_sum = Z[idx].sum(0)
            n = len(idx)
            for row in idx:
                Zc[row] = (Z[row] - (group_sum - Z[row]) / max(n - 1, 1)) * ((n - 1) / n)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        LOGIT[twin_rows] += mc["twin_w"] * np.log(np.clip(softmax(centred), 1e-9, 1))
    del Z
    gc.collect()
    print(f"{ckpt_name} done ({time.time() - T0:.0f}s)", flush=True)


topic clusters: 3 singletons, 0 passages in clusters of 2-18


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
Key                                                    | Status     |  | 
-------------------------------------------------------+------------+--+-
model.layers.{24...35}.self_attn.v_proj.weight         | UNEXPECTED |  | 
model.layers.{24...35}.self_attn.o_proj.weight         | UNEXPECTED |  | 
model.layers.{24...35}.mlp.up_proj.weight              | UNEXPECTED |  | 
model.layers.{24...35}.self_attn.k_proj.weight         | UNEXPECTED |  | 
model.layers.{24...35}.self_attn.q_norm.weight         | UNEXPECTED |  | 
model.layers.{24...35}.mlp.gate_proj.weight            | UNEXPECTED |  | 
model.layers.{24...35}.self_attn.q_proj.weight         | UNEXPECTED |  | 
model.layers.{24...35}.input_layernorm.weight          | UNEXPECTED |  | 
model.layers.{24...35}.mlp.down_proj.weight            | UNEXPECTED |  | 
model.layers.{24...35}.self_attn.k_norm.weight         | UNEXPECTED |  | 
model.layers.{

lora8b_v1 done (356s)


Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen-3/transformers/14b/1
Key                                                    | Status     |  | 
-------------------------------------------------------+------------+--+-
model.layers.{27...39}.self_attn.o_proj.weight         | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.k_proj.weight         | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.q_norm.weight         | UNEXPECTED |  | 
model.layers.{27...39}.mlp.gate_proj.weight            | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.q_proj.weight         | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.k_norm.weight         | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.v_proj.weight         | UNEXPECTED |  | 
model.layers.{27...39}.mlp.down_proj.weight            | UNEXPECTED |  | 
model.layers.{27...39}.input_layernorm.weight          | UNEXPECTED |  | 
model.layers.{27...39}.mlp.up_proj.weight              | UNEXPECTED |  | 
model.layers.

lora14b_v1 done (542s)


## 4. Blend, then balance

The members are combined by adding their weighted log-probabilities, which amounts to multiplying their probabilities.

One last fix. Our models learned partly from the public pool, where some models have 2,700 passages and others 1,100, so they carry slightly wrong priors. The test set, like train, should hold roughly equal numbers of each model. **Sinkhorn balancing** rescales each class's column until every class gets about 1/18 of the probability mass, then renormalises each row, repeated 100 times. It uses no labels. Locally it is worth about +0.5 points, on a test set that is far from balanced it would hurt.


In [ ]:
# Conservative label-specific calibration and sibling separation.
# Biases come from the earlier prompt-grouped OOF recall gaps and are tightly clipped.
OOF_RECALL = {
    "claude-fable-5": 0.0738, "gpt-5.6-sol": 0.1200,
    "glm-5.3": 0.1333, "glm-5.3-flash": 0.1400,
    "gpt-5.6-luna": 0.2533, "gpt-5.6-terra": 0.2600,
    "gemini-3.8-flash": 0.3000, "gemini-3.7-flash": 0.3400,
}
mean_recall = 0.3389
for lab, rec in OOF_RECALL.items():
    LOGIT[:, LABELS.index(lab)] += float(np.clip(0.10 * np.log(mean_recall / rec), -0.12, 0.12))

SIBLING_PAIRS = [
    ("gemini-3.7-flash", "gemini-3.8-flash"),
    ("gemini-3.6-flash", "gemini-3.7-flash"),
    ("gpt-5.6-luna", "gpt-5.6-sol"),
    ("gpt-5.6-luna", "gpt-5.6-terra"),
    ("gpt-5.6-sol", "gpt-5.6-terra"),
]
pair_vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2,
                           max_features=160000, sublinear_tf=True, lowercase=False,
                           dtype=np.float32)
X_pair_train = pair_vec.fit_transform(train.text.astype(str))
X_pair_test = pair_vec.transform(test.text.astype(str))
base_order = np.argsort(LOGIT, axis=1)[:, -2:]
base_margin = LOGIT.max(1) - np.partition(LOGIT, -2, axis=1)[:, -2]
for left, right in SIBLING_PAIRS:
    il, ir = LABELS.index(left), LABELS.index(right)
    ambiguous = np.isin(base_order, [il, ir]).all(1) & (base_margin < 0.28)
    if not ambiguous.any():
        continue
    keep = np.isin(y, [il, ir])
    pair_model = LinearSVC(C=0.35, class_weight="balanced", max_iter=3000, random_state=17)
    pair_model.fit(X_pair_train[keep], y[keep])
    score = pair_model.decision_function(X_pair_test[ambiguous])
    signed = np.tanh(np.clip(score, -2.0, 2.0)) * 0.16
    for j, row in enumerate(np.flatnonzero(ambiguous)):
        if pair_model.classes_[1] == ir:
            LOGIT[row, ir] += signed[j]; LOGIT[row, il] -= signed[j]
        else:
            LOGIT[row, il] += signed[j]; LOGIT[row, ir] -= signed[j]
# Family-aware specialist gate. Apply only when the main blend is uncertain
# across families; this avoids overriding confident model-level predictions.
FAMILY_OF = {}
for lab in LABELS:
    if lab.startswith("gpt-"):
        FAMILY_OF[lab] = "OpenAI"
    elif lab.startswith("gemini-"):
        FAMILY_OF[lab] = "Google"
    elif lab.startswith("claude-"):
        FAMILY_OF[lab] = "Anthropic"
    elif lab.startswith("qwen"):
        FAMILY_OF[lab] = "Qwen"
    elif lab.startswith("glm-"):
        FAMILY_OF[lab] = "ZAI"
    elif lab.startswith("kimi-"):
        FAMILY_OF[lab] = "Moonshot"
FAMILY_NAMES = ["OpenAI", "Google", "Anthropic", "Qwen", "ZAI", "Moonshot"]
family_y = np.array([FAMILY_NAMES.index(FAMILY_OF[lab]) for lab in train.model.astype(str)])
family_model = LinearSVC(C=0.20, class_weight="balanced", max_iter=3000, random_state=23)
family_model.fit(X_pair_train, family_y)
family_p = softmax(family_model.decision_function(X_pair_test))
main_order = np.argsort(LOGIT, axis=1)[:, -2:]
main_margin = LOGIT.max(1) - np.partition(LOGIT, -2, axis=1)[:, -2]
main_family = np.array([FAMILY_NAMES.index(FAMILY_OF[LABELS[i]]) for i in LOGIT.argmax(1)])
runner_family = np.array([FAMILY_NAMES.index(FAMILY_OF[LABELS[i]]) for i in main_order[:, -2]])
ambiguous_cross_family = (main_family != runner_family) & (main_margin < 0.55) & (family_p.max(1) > 0.60)
if ambiguous_cross_family.any():
    for li, lab in enumerate(LABELS):
        fi = FAMILY_NAMES.index(FAMILY_OF[lab])
        LOGIT[ambiguous_cross_family, li] += 0.08 * np.log(np.clip(family_p[ambiguous_cross_family, fi], 1e-6, 1.0))
print(f"family gate applied to {int(ambiguous_cross_family.sum())} ambiguous rows", flush=True)

# Hierarchical family specialist: once the family is reliable, let a local
# classifier resolve its sibling models. It only contributes on low-margin
# rows, so it cannot disturb confident cross-family decisions.
hier_rows = (family_p.max(1) > 0.60) & (main_margin < 0.55)
hier_count = 0
for fi, fam in enumerate(FAMILY_NAMES):
    rows = np.flatnonzero(hier_rows & (family_p.argmax(1) == fi))
    tr = np.flatnonzero(family_y == fi)
    local_labels = np.unique(y[tr])
    if len(rows) == 0 or len(local_labels) < 2:
        continue
    local_model = LinearSVC(C=0.35, class_weight="balanced", max_iter=3000, random_state=41 + fi)
    local_model.fit(X_pair_train[tr], y[tr])
    local_prob = softmax(local_model.decision_function(X_pair_test[rows]))
    for j, lab in enumerate(local_model.classes_):
        LOGIT[rows, lab] += 0.10 * np.log(np.clip(local_prob[:, j], 1e-9, 1))
    hier_count += len(rows)
print(f"hierarchical family specialist applied to {hier_count} rows", flush=True)
del X_pair_train, X_pair_test, pair_vec, family_model


In [5]:
P = softmax(LOGIT)
if len(test) >= 50:   # skip on the 3-row public test.csv
    # Damped marginal balancing: correct public-pool prior skew without forcing
    # an abrupt uniform test distribution onto every prediction.
    alpha = 0.50
    target = np.full((1, K), len(P) / K, dtype=np.float64)
    for _ in range(40):
        column_mass = P.sum(0, keepdims=True).clip(min=1e-12)
        P *= (target / column_mass) ** alpha
        P /= P.sum(1, keepdims=True).clip(min=1e-12)
    P /= P.sum(1, keepdims=True).clip(min=1e-12)

# Empirical orthographic shield learned from official training passages.
# Apply after marginal balancing so impossible-family evidence cannot be
# resurrected by the balancing step. Keep factors soft and renormalize rows.
CLAUDE_GLM_IDX = [i for i, lab in enumerate(LABELS)
                  if lab.startswith("claude-") or lab.startswith("glm-")]
GEMINI_IDX = [i for i, lab in enumerate(LABELS) if lab.startswith("gemini-")]
GPT_SOL_IDX = LABELS.index("gpt-5.6-sol")
CURLY_RE = re.compile(r"[\u2018\u2019\u201c\u201d]")
for i, txt in enumerate(test["text"].astype(str).tolist()):
    if CURLY_RE.search(txt):
        P[i, CLAUDE_GLM_IDX] *= 0.01
    if "—" in txt:
        P[i, GEMINI_IDX] *= 0.05
    if "'" in txt:
        P[i, GPT_SOL_IDX] *= 0.01
P /= P.sum(1, keepdims=True).clip(min=1e-12)
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")


model
claude-opus-4-8           1
gpt-5.6-sol               1
gemini-3.1-pro-preview    1
done in 542s
